# Nepali Student Assistant — Intent Classification (3 Models)

This notebook trains and compares **three classification techniques** on a Nepali educational question dataset:

- Logistic Regression
- Linear SVM
- K-Nearest Neighbors

It also produces figures/metrics you can paste into the IEEE report.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
CODES_DIR = PROJECT_ROOT / "codes"
import sys
sys.path.insert(0, str(CODES_DIR))
from config import CLEAN_XL, OUTPUTS_DIR

print("Project root:", PROJECT_ROOT)
print("Clean dataset path:", CLEAN_XL)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cleaned dataset
df = pd.read_excel(CLEAN_XL)
df = df.dropna(subset=["question", "intent"])
df["question"] = df["question"].astype(str)
df["intent"] = df["intent"].astype(str)

display(df.head())
print("Rows:", len(df))
print(df["intent"].value_counts())

In [ ]:
X = df["question"].values
y = df["intent"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

## Model definitions

We use a strong and simple text baseline: **TF-IDF character n-grams** (works well for Nepali morphology and spelling variation) + a classifier.

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    sublinear_tf=True,
    min_df=2,
)

models = {
    "LogReg": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=None,
    ),
    "LinearSVM": LinearSVC(
        class_weight="balanced",
        max_iter=5000,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
    ),
}

pipelines = {name: Pipeline([( "tfidf", vectorizer), ("clf", clf)]) for name, clf in models.items()}
list(pipelines.keys())

In [ ]:
reports = {}

for name, pipe in pipelines.items():
    print("\n===", name, "===")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rep = classification_report(y_test, pred, output_dict=True, zero_division=0)
    reports[name] = rep
    print(classification_report(y_test, pred, zero_division=0))

In [ ]:
# Summarize metrics into a compact dataframe
rows = []
for name, rep in reports.items():
    rows.append({
        "model": name,
        "accuracy": rep["accuracy"],
        "macro_f1": rep["macro avg"]["f1-score"],
        "weighted_f1": rep["weighted avg"]["f1-score"],
    })
metrics_df = pd.DataFrame(rows).sort_values("weighted_f1", ascending=False)
display(metrics_df)

metrics_out = OUTPUTS_DIR / "three_model_metrics.csv"
metrics_df.to_csv(metrics_out, index=False)
print("Saved:", metrics_out)

In [ ]:
# Confusion matrix for the best model (by weighted F1)
best_name = metrics_df.iloc[0]["model"]
best_pipe = pipelines[best_name]
best_pred = best_pipe.predict(X_test)

labels = sorted(np.unique(y))
cm = confusion_matrix(y_test, best_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title(f"Confusion Matrix — {best_name}")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

cm_out = OUTPUTS_DIR / f"confusion_matrix_{best_name}.png"
plt.savefig(cm_out, dpi=200)
print("Saved:", cm_out)
plt.show()